In [1]:
pip install beautifulsoup4 contractions

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.8.4-py3-none-any.whl.metadata (4.6 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached soupsieve-2.8.4-py3-none-any.whl (37 kB)
   ---------------------------------------- 0.0/345.1 kB ? eta -:--:--
   --- ------------------------------------ 30.7/345.1 kB 1.3 MB/s eta 0:00:01
   ------ -------------------------------- 61.4/345.1 kB 825.8 kB/s eta 0:00:01
   ----------- -------------------------- 102.4/345.1 kB 845.5 kB/s eta 0:00:01
   -------------- ----------------------- 133.1/345.1 kB 787.7 kB/s eta 0:00:01
   -------------------- ----------------- 184.3/345.1 kB 794.9 kB/s eta 0:00:01
   ------------------------- ------------ 235.5/345.1 kB 846.9 kB/s eta 0:00:01
   ------------------------------- ------ 286.7/345.1 kB 886.2 kB/s eta 0:00:01
   -------------------------------------  337.9/345.1 kB 952.6 kB/s eta 0:00:01
   ----------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
df = pd.read_csv("../data/amazon_reviews_cleaned.csv")
print(df.shape)
df.head()

(65229, 5)


,rating,title,text,sentiment,text_length
0,5.0,Great pictures and recipes,Wonderful recipes in this magazine.,positive,5
1,4.0,great for kids who love sports!,Great sports magazine that's on my 9 year olds...,positive,10
2,5.0,A great look at what's new on the kosher scene...,"""Joy of Kosher"" magazine fills a much-needed n...",positive,532
3,5.0,"If you enjoy reading an eye-catching, easy-to-...",I've been addicted to Martha Stewart's Everyda...,positive,324
4,1.0,Too many “ gear ads” !!,Too many ads!,negative,3


In [3]:
from bs4 import BeautifulSoup

def strip_html(text):
    return BeautifulSoup(str(text), "html.parser").get_text(separator=' ')

sample = df[df['text'].str.contains('<br', case=False, na=False)]['text'].iloc[0]
print("BEFORE:\n", sample[:300])
print("\nAFTER:\n", strip_html(sample)[:300])

BEFORE:
 "Joy of Kosher" magazine fills a much-needed niche for kosher recipes, cookbook reviews, and the kosher lifestyle not found in other mainstream magazines. Sporting beautifully-photographed covers and spreads that are equally at home next to powerhouse food and cooking magazines, "Joy of Kosher" (whi

AFTER:
 "Joy of Kosher" magazine fills a much-needed niche for kosher recipes, cookbook reviews, and the kosher lifestyle not found in other mainstream magazines. Sporting beautifully-photographed covers and spreads that are equally at home next to powerhouse food and cooking magazines, "Joy of Kosher" (whi


In [5]:
import contractions

test = "I don't like this, it's not what I expected and wasn't worth it."
print(contractions.fix(test))

I do not like this, it is not what I expected and was not worth it.


In [6]:
import re
import contractions
from bs4 import BeautifulSoup

def clean_text(text):
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(separator=' ')  # strip HTML
    text = contractions.fix(text)                                       # expand contractions
    text = text.lower()                                                 # lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)                                # remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()                             # collapse whitespace
    return text

# Test on a few samples
for t in df['text'].sample(3, random_state=1):
    print("BEFORE:", t[:150])
    print("AFTER:", clean_text(t)[:150])
    print()

BEFORE: As a standard cooking magazine it contains some interesting and some not too interesting recipes and articles. The recipes are not on the higher level
AFTER: as a standard cooking magazine it contains some interesting and some not too interesting recipes and articles the recipes are not on the higher level 

BEFORE: This is a wonderful magazine that represents many breeds of dogs and many types of hunting. I would recommend it to anyone who is interested in huntin
AFTER: this is a wonderful magazine that represents many breeds of dogs and many types of hunting i would recommend it to anyone who is interested in hunting

BEFORE: Good read, good info
AFTER: good read good info



In [7]:
df['text_clean'] = df['text'].apply(clean_text)
print(df[['text', 'text_clean']].sample(3, random_state=2))

                                                    text  \
47725                                      To Many adds!   
22486                         some good science in there   
21968  This magazine is fun and upbeat. The pictures ...   

                                              text_clean  
47725                                       to many adds  
22486                         some good science in there  
21968  this magazine is fun and upbeat the pictures a...  


In [9]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\satya\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\satya\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\satya\AppData\Roaming\nltk_data...


True

In [10]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_for_tfidf(text):
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# Test
for t in df['text_clean'].sample(3, random_state=1):
    print("INPUT:", t[:150])
    print("TFIDF-READY:", clean_for_tfidf(t)[:150])
    print()

INPUT: as a standard cooking magazine it contains some interesting and some not too interesting recipes and articles the recipes are not on the higher level 
TFIDF-READY: standard cooking magazine contains interesting interesting recipe article recipe higher level bon appetite late gourmet magazine lately magazine resem

INPUT: this is a wonderful magazine that represents many breeds of dogs and many types of hunting i would recommend it to anyone who is interested in hunting
TFIDF-READY: wonderful magazine represents many breed dog many type hunting would recommend anyone interested hunting dog look foward recieving every month hardly 

INPUT: good read good info
TFIDF-READY: good read good info



In [11]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))

# Keep negation words — critical for sentiment
negation_words = {'not', 'no', 'nor', 'never', "n't"}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def clean_for_tfidf(text):
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# Re-test
for t in df['text_clean'].sample(3, random_state=1):
    print("INPUT:", t[:150])
    print("TFIDF-READY:", clean_for_tfidf(t)[:150])
    print()

INPUT: as a standard cooking magazine it contains some interesting and some not too interesting recipes and articles the recipes are not on the higher level 
TFIDF-READY: standard cooking magazine contains interesting not interesting recipe article recipe not higher level bon appetite late gourmet magazine lately magazi

INPUT: this is a wonderful magazine that represents many breeds of dogs and many types of hunting i would recommend it to anyone who is interested in hunting
TFIDF-READY: wonderful magazine represents many breed dog many type hunting would recommend anyone interested hunting dog look foward recieving every month hardly 

INPUT: good read good info
TFIDF-READY: good read good info



In [12]:
df['text_tfidf'] = df['text_clean'].apply(clean_for_tfidf)
print(df[['text', 'text_clean', 'text_tfidf']].sample(3, random_state=3))

                                                    text  \
19678  [Harpers began before the Civil War and is sti...   
36563  The pages where they pretty much plan trips fo...   
7975   Printed on premium quality paper with high-res...   

                                              text_clean  \
19678  harpers began before the civil war and is stil...   
36563  the pages where they pretty much plan trips fo...   
7975   printed on premium quality paper with high res...   

                                              text_tfidf  
19678  harper began civil war still wonderful read wh...  
36563  page pretty much plan trip best helped expand ...  
7975   printed premium quality paper high resolution ...  


In [13]:
df.to_csv("../data/amazon_reviews_processed.csv", index=False)
print("Saved:", df.shape)
print(df.columns.tolist())

Saved: (65229, 7)
['rating', 'title', 'text', 'sentiment', 'text_length', 'text_clean', 'text_tfidf']
